# 02 — Annotation Audit

## Objective

Normalize LabelMe circles and polygons, derive bounding boxes, and identify invalid or review-worthy annotation geometry.

## Motivation

Downstream analyses require a consistent geometric representation. Errors must be separated from warnings so that valid cases, such as partially visible coins, are not incorrectly rejected.

## Inputs

- `curation/outputs/01-dataset-audit/inventory.csv`
- LabelMe JSON files from the configured annotation directory.

## Outputs

- `curation/outputs/02-annotation-audit/annotations_normalized.csv`
- `curation/outputs/02-annotation-audit/annotation_findings.csv`
- `curation/outputs/02-annotation-audit/manifest.yaml`


In [ ]:
# Environment-specific setup
import pathlib
import sys

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../../')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')

ROOT = base_folder.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
from curation.common import load_config, prepare_dataset, stage_output_dir

CONFIG = load_config(ROOT)
DATASET = prepare_dataset(ROOT)
IMAGES_DIR = DATASET["images_dir"]
ANNOTATIONS_DIR = DATASET["annotations_dir"]

from curation.common import load_labelme, normalize_labelme_shapes, write_manifest
import pandas as pd

STAGE = "02-annotation-audit"
STAGE_DIR = stage_output_dir(STAGE, ROOT)
INVENTORY_PATH = stage_output_dir("01-dataset-audit", ROOT, create=False) / "inventory.csv"
if not INVENTORY_PATH.is_file():
    raise FileNotFoundError("Run 01-dataset-audit.ipynb first.")

In [ ]:
tolerance = float(CONFIG["audits"]["annotation"]["boundary_tolerance_px"])
inventory = pd.read_csv(INVENTORY_PATH)
records, findings = [], []
matched = inventory[inventory["annotation_relative_path"].notna() & inventory["readable"]]
for row in matched.itertuples(index=False):
    value = load_labelme(ANNOTATIONS_DIR / row.annotation_relative_path)
    normalized, current_findings = normalize_labelme_shapes(
        value,
        relative_path=row.relative_path,
        annotation_relative_path=row.annotation_relative_path,
        image_width=int(row.width),
        image_height=int(row.height),
        boundary_tolerance_px=tolerance,
    )
    records.extend(normalized)
    findings.extend(current_findings)

annotations = pd.DataFrame(records)
finding_columns = ["relative_path", "annotation_relative_path", "shape_index", "label", "shape_type", "severity", "finding_type", "detail"]
annotation_findings = pd.DataFrame(findings, columns=finding_columns)

duplicate_mask = annotations.assign(
    _x1=annotations["bbox_xmin"].round(3), _y1=annotations["bbox_ymin"].round(3),
    _x2=annotations["bbox_xmax"].round(3), _y2=annotations["bbox_ymax"].round(3),
).duplicated(["relative_path", "label", "shape_type", "_x1", "_y1", "_x2", "_y2"], keep=False)
for row in annotations[duplicate_mask].itertuples(index=False):
    annotation_findings.loc[len(annotation_findings)] = [
        row.relative_path, row.annotation_relative_path, row.shape_index, row.label,
        row.shape_type, "warning", "duplicate_geometry_candidate",
        "Another shape has the same rounded label, type, and bounding box.",
    ]

annotations_path = STAGE_DIR / "annotations_normalized.csv"
findings_path = STAGE_DIR / "annotation_findings.csv"
annotations.to_csv(annotations_path, index=False)
annotation_findings.to_csv(findings_path, index=False)
display(annotations.head())
display(annotation_findings)

In [ ]:
write_manifest(
    STAGE,
    "02-annotation-audit.ipynb",
    inputs={"inventory": INVENTORY_PATH},
    parameters={"boundary_tolerance_px": tolerance, "supported_shapes": ["circle", "polygon"]},
    artifacts=[annotations_path, findings_path],
    summary={
        "normalized_annotations": int(len(annotations)),
        "errors": int((annotation_findings["severity"] == "error").sum()),
        "warnings": int((annotation_findings["severity"] == "warning").sum()),
        "labels": {str(k): int(v) for k, v in annotations["label"].value_counts().sort_index().items()},
        "shape_types": {str(k): int(v) for k, v in annotations["shape_type"].value_counts().sort_index().items()},
    },
    repo_root=ROOT,
)